# 01 · Baseline sanity check (S0, 100% labels, seed 0)

**Runtime → Change runtime type → T4 GPU.** Then **Runtime → Run all**.

Goal (`docs/training_protocol_v1.md`): fold-10 macro-AUROC over all PTB-XL statements near the published
xresnet1d50 value (≈ 0.92–0.93). Below ~0.90 means the pipeline needs debugging before any SSL work.

If the session disconnects, **Run all** again: the run resumes from its last checkpoint on Drive.
Do not use *File → Save a copy in GitHub*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, shutil, zipfile
PROJECT = '/content/drive/MyDrive/robust-ecg-ssl'
CACHE_DRIVE, RUNS, RAW = f'{PROJECT}/cache', f'{PROJECT}/runs', f'{PROJECT}/raw'
CACHE = '/content/cache'                      # local copy: much faster reads than Drive
REPO = '/content/robust-ecg-ssl'
os.makedirs(RUNS, exist_ok=True)
if not os.path.exists(REPO):
    !git clone -q https://github.com/Salako07/robust-ecg-ssl.git {REPO}
!git -C {REPO} pull -q && git -C {REPO} log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# scp_statements.csv is needed in the cache (older caches were built without it)
if not os.path.exists(f'{CACHE_DRIVE}/scp_statements.csv'):
    with zipfile.ZipFile(f'{RAW}/ptbxl-1.0.3.zip') as z:
        name = [n for n in z.namelist() if n.endswith('scp_statements.csv')][0]
        with z.open(name) as src, open(f'{CACHE_DRIVE}/scp_statements.csv', 'wb') as dst:
            shutil.copyfileobj(src, dst)
    print('added scp_statements.csv to the Drive cache')

# copy the cache to local disk once per session (~1-3 min)
os.makedirs(CACHE, exist_ok=True)
for f in os.listdir(CACHE_DRIVE):
    s, d = f'{CACHE_DRIVE}/{f}', f'{CACHE}/{f}'
    if os.path.isfile(s) and (not os.path.exists(d) or os.path.getsize(d) != os.path.getsize(s)):
        shutil.copy(s, d)
!ls -la {CACHE}

In [ ]:
# The sanity-check run. Prints validation macro-AUROC 50 times; expect roughly 30-60 min on a T4.
!python {REPO}/scripts/train.py --arm S0 --budget 1.0 --seed 0 --cache {CACHE} --runs {RUNS} --workers 2

In [ ]:
import pandas as pd, json
r = json.load(open(f'{RUNS}/S0_b1_s0_lik0/done.json'))
keys = ['test_macro_auroc', 'test_n_labels', 'best_val_macro_auroc', 'best_step', 'total_steps',
        'test_e3_macro_auroc', 'sph_e3_macro_auroc', 'e3_degradation', 'train_minutes_last_session']
for k in keys: print(f'{k:28s} {r[k]}')
print()
print({k[len('sph_auroc_'):]: round(v, 3) for k, v in r.items() if k.startswith('sph_auroc_')})
print(open(f'{RUNS}/S0_b1_s0_lik0/log.csv').read()[-800:])

**Paste the output of the last cell into the chat.** It is the first real result of the project.